In [1]:
from incidentiq.pipeline import IncidentIQ
from incidentiq.evaluation.cases import EVALUATION_CASES

DATA_PATH = "../data/processed/logs.parquet"

incidentiq = IncidentIQ(
    data_path=DATA_PATH,
    model="gemini-3.5-flash-lite",
)

print(f"Loaded {len(EVALUATION_CASES)} evaluation cases.")

e:\incidentiq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 63/63 [00:04<00:00, 13.82it/s]


Loaded 5 evaluation cases.


In [2]:
case = EVALUATION_CASES[0]

print("Query:", case["query"])
print("Initial top_k:", case["top_k"])

Query: machine check timeout
Initial top_k: 3


In [3]:
# result = incidentiq.investigate(
#     query=case["query"],
#     top_k=case["top_k"],
# )

In [4]:
# print("Query:", result.query)
# print("Initial evidence:", len(result.evidence))
# print("Tool evidence:", len(result.tool_evidence))
# print("Grounded:", result.grounding.is_grounded)
# print("Grounding violations:", len(result.grounding.violations))

In [5]:
# evaluation_record = {
#     "query": result.query,
#     "initial_evidence_count": len(result.evidence),
#     "tool_evidence_count": len(result.tool_evidence),
#     "grounded": result.grounding.is_grounded,
#     "grounding_violations": len(result.grounding.violations),
# }

# evaluation_record

In [6]:
from incidentiq.evaluation.expected import EXPECTED_FINDINGS


def evaluate_findings(result):
    expected = EXPECTED_FINDINGS[result.query]
    analysis = result.analysis

    observation_count = len(analysis.observations)
    unknown_count = len(analysis.unknowns)

    observation_score = min(
        observation_count / expected["required_observations"],
        1.0,
    )

    unknown_score = min(
        unknown_count / expected["required_unknowns"],
        1.0,
    )

    coverage = (observation_score + unknown_score) / 2

    return {
        "finding_coverage": coverage,
        "observation_count": observation_count,
        "unknown_count": unknown_count,
        "missing_observations": max(
            expected["required_observations"] - observation_count,
            0,
        ),
        "missing_unknowns": max(
            expected["required_unknowns"] - unknown_count,
            0,
        ),
    }

In [7]:
def evaluate_case(case):
    result = incidentiq.investigate(
        query=case["query"],
        top_k=case["top_k"],
    )

    initial_ids = {item["doc_id"] for item in result.evidence}
    tool_ids = {item["doc_id"] for item in result.tool_evidence}
    new_evidence_ids = tool_ids - initial_ids

    finding_eval = evaluate_findings(result)

    return {
        "query": result.query,
        "result": result,
        "tool_calls": result.tool_calls,
        "tool_call_count": len(result.tool_calls),
        "initial_evidence_count": len(result.evidence),
        "tool_evidence_count": len(result.tool_evidence),
        "unique_tool_evidence_count": len(tool_ids),
        "new_evidence_count": len(new_evidence_ids),
        "grounded": result.grounding.is_grounded,
        "grounding_violations": len(result.grounding.violations),
        "finding_coverage": finding_eval["finding_coverage"],
        "observation_count": finding_eval["observation_count"],
        "unknown_count": finding_eval["unknown_count"],
        "missing_observations": finding_eval["missing_observations"],
        "missing_unknowns": finding_eval["missing_unknowns"],
    }

In [8]:
results = []

for case in EVALUATION_CASES:
    print(f"Running: {case['query']}")
    
    evaluation = evaluate_case(case)
    results.append(evaluation)
    
    print(evaluation)
    print()

Running: machine check timeout
FUNCTION NAME: 'search_logs'
FUNCTION NAME TYPE: <class 'str'>
EXPECTED: 'search_logs'
EQUAL: True
Tool call : search_logs with arguments: {'query': 'R26-M0-N0-I:J18-U11'}
Tool returned 10 results.
Total tool evidence :  10
Tool evidence IDs:  [1795, 294, 1728, 1647, 1303, 1617, 1650, 1296, 1623, 1335]
Prompt: 0.000s | Gemini API: 10.524s | Parsing: 0.000s
Retrieval: 0.011s | Context: 0.001s | Patterns: 0.000s | Reasoning: 10.524s | Total: 10.536s
{'query': 'machine check timeout', 'result': InvestigationResult(query='machine check timeout', analysis=IncidentAnalysis(summary='Node R26-M0-N0-I:J18-U11 experienced repeated MACHINE CHECK DCR read timeout events over a 14-minute interval on November 21, 2005.', observations=[Observation(statement="Node R26-M0-N0-I:J18-U11 logged 'MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)' three times on 2005-11-21 at 06:41:29, 06:48:15, and 06:56:03.", evidence_ids=[1769, 1770, 1771])], hypotheses=

In [9]:
for res in results:
    print(res)

{'query': 'machine check timeout', 'result': InvestigationResult(query='machine check timeout', analysis=IncidentAnalysis(summary='Node R26-M0-N0-I:J18-U11 experienced repeated MACHINE CHECK DCR read timeout events over a 14-minute interval on November 21, 2005.', observations=[Observation(statement="Node R26-M0-N0-I:J18-U11 logged 'MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)' three times on 2005-11-21 at 06:41:29, 06:48:15, and 06:56:03.", evidence_ids=[1769, 1770, 1771])], hypotheses=[], unknowns=['What specific hardware component or device bus caused the DCR read timeout?', 'Whether this node experienced subsequent fatal failures or required a hardware replacement/reboot.'], next_steps=[]), evidence=[{'doc_id': 1770, 'timestamp': Timestamp('2005-11-21 06:48:15.351050'), 'node': 'R26-M0-N0-I:J18-U11', 'severity': 'INFO', 'message': 'MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)'}, {'doc_id': 1769, 'timestamp': Timestamp('2005-11-21 06

In [10]:
import pandas as pd

df_results = pd.DataFrame(results)

df_results

,query,result,tool_calls,tool_call_count,initial_evidence_count,tool_evidence_count,unique_tool_evidence_count,new_evidence_count,grounded,grounding_violations,finding_coverage,observation_count,unknown_count,missing_observations,missing_unknowns
0,machine check timeout,query='machine check timeout' analysis=Inciden...,[R26-M0-N0-I:J18-U11],1,3,10,10,10,True,0,0.75,1,2,1,0
1,floating point exception,query='floating point exception' analysis=Inci...,[floating point alignment exceptions],1,3,5,5,5,True,0,1.00,4,3,0,0
2,node card failure,query='node card failure' analysis=IncidentAna...,"[Node card is not fully functional, PGOOD IS N...",2,3,20,19,16,True,0,1.00,3,2,0,0
3,cache parity error,query='cache parity error' analysis=IncidentAn...,[R02-M1-N0-C:J12-U11],1,3,10,10,10,True,0,1.00,2,2,0,0
4,network packet error,query='network packet error' analysis=Incident...,[Error receiving packet on tree network expect...,1,3,10,10,7,True,0,1.00,3,3,0,0


In [11]:
for item in results:
    result = item["result"]

    print("=" * 80)
    print("QUERY:", result.query)
    print("SUMMARY:", result.analysis.summary)

    print("\nOBSERVATIONS:")
    for obs in result.analysis.observations:
        print("-", obs.statement)
        print("  evidence:", obs.evidence_ids)

    print("\nHYPOTHESES:")
    for hyp in result.analysis.hypotheses:
        print("-", hyp.statement)
        print("  confidence:", hyp.confidence)
        print("  evidence:", hyp.evidence_ids)

    print("\nUNKNOWNS:")
    for unknown in result.analysis.unknowns:
        print("-", unknown)

QUERY: machine check timeout
SUMMARY: Node R26-M0-N0-I:J18-U11 experienced repeated MACHINE CHECK DCR read timeout events over a 14-minute interval on November 21, 2005.

OBSERVATIONS:
- Node R26-M0-N0-I:J18-U11 logged 'MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)' three times on 2005-11-21 at 06:41:29, 06:48:15, and 06:56:03.
  evidence: [1769, 1770, 1771]

HYPOTHESES:

UNKNOWNS:
- What specific hardware component or device bus caused the DCR read timeout?
- Whether this node experienced subsequent fatal failures or required a hardware replacement/reboot.
QUERY: floating point exception
SUMMARY: Multiple compute nodes across different racks report '8 floating point alignment exceptions' logged with INFO severity between 2005-07-20 16:03:16 and 2005-07-20 16:19:01.

OBSERVATIONS:
- Node R15-M0-N2-C:J04-U11 logged '8 floating point alignment exceptions' with INFO severity at 2005-07-20 16:11:50.364727.
  evidence: [1040]
- Node R11-M1-NB-C:J15-U11 logged '8 floa